In [1]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated Successfully')

Authenticated Successfully


In [2]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='crafty-airfoil-382608')

In [5]:
query_string = """
WITH CTE_First AS (
  SELECT *, ROUND(AVG(Amount) OVER(PARTITION BY Customer_ID ORDER BY Transaction_Timestamp ROWS BETWEEN 10 PRECEDING AND 1 PRECEDING), 2) AS Rolling_Avg
  FROM `Fraud_Detection.Fact_Transactions`
)
SELECT
  *
FROM
  CTE_First
WHERE
  Amount > (Rolling_Avg * 5)
"""

fraud_report_df = client.query(query_string).to_dataframe()

if fraud_report_df.empty:
    print("No fraud detected today. System Green.")
else:
    print(f"ALARM: {len(fraud_report_df)} suspicious transactions found!")
    display(fraud_report_df.head())

ALARM: 11 suspicious transactions found!


,Transaction_ID,Customer_ID,Transaction_Timestamp,Amount,Category,Rolling_Avg
0,5827,28,2026-04-16 01:35:01.060989+00:00,447.39,Dining,54.75
1,5633,34,2026-04-16 04:49:01.060989+00:00,413.89,Pharmacy,76.09
2,9997,42,2026-04-19 10:01:59.060989+00:00,18000.00,Crypto Exchange,274.85
3,9998,42,2026-04-19 10:02:00.060989+00:00,12000.00,Online Gambling,2069.32
4,5854,55,2026-04-16 01:08:01.060989+00:00,88.36,Online Shopping,16.63


In [6]:
file_name = "Daily_Fraud_Alert_Report.csv"
fraud_report_df.to_csv(file_name, index=False)

print(f"Report generated: {file_name}")

from google.colab import files
files.download(file_name)

Report generated: Daily_Fraud_Alert_Report.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>